# Task 7: Time-Based Drift Simulation
**Assignment 4 — IEEE CIS Fraud Detection**

Strategy:
- Split data on `TransactionDT` quartiles: Q1+Q2 = training, Q3 = validation, Q4 = "future"
- Train XGBoost on Q1+Q2, evaluate on Q3 then Q4
- Inject a synthetic new fraud pattern into Q4
- Measure performance degradation
- Retrain on Q4 and compare feature importance shift

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.ingest import load_sample
from src.data.preprocess import preprocess
from src.drift.time_split import quartile_split, inject_new_fraud_pattern
from src.drift.detect import compute_drift, compare_feature_importance

import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report

print('Libraries loaded')

In [ ]:
# Load the working sample
df = load_sample()
print(f'Dataset: {df.shape}  | Fraud rate: {df["isFraud"].mean():.2%}')

In [ ]:
# Temporal quartile split
q12, q3, q4 = quartile_split(df, time_col='TransactionDT')

# Visualise fraud rate across time windows
windows = ['Q1+Q2 (train)', 'Q3 (val)', 'Q4 (future)']
rates = [q12['isFraud'].mean(), q3['isFraud'].mean(), q4['isFraud'].mean()]
plt.figure(figsize=(8, 4))
plt.bar(windows, rates, color=['steelblue', 'orange', 'red'])
plt.ylabel('Fraud Rate')
plt.title('Fraud Rate by Time Window')
plt.tight_layout()
plt.savefig('../outputs/drift/fraud_rate_by_window.png', dpi=150)
plt.show()

In [ ]:
# Preprocess: fit on Q12, apply to Q3 and Q4
X_train, y_train, enc, med = preprocess(q12, fit=True)
X_q3, y_q3, _, _           = preprocess(q3,  encoders=enc, medians=med, fit=False)
X_q4, y_q4, _, _           = preprocess(q4,  encoders=enc, medians=med, fit=False)

print(f'Train: {X_train.shape}  Q3: {X_q3.shape}  Q4: {X_q4.shape}')

In [ ]:
# Train on Q1+Q2
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
model_q12 = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    scale_pos_weight=neg/pos, tree_method='hist',
    eval_metric='logloss', random_state=42, n_jobs=-1
)
model_q12.fit(X_train.values, y_train.values,
              eval_set=[(X_q3.values, y_q3.values)], verbose=50)
print('Q12 model trained')

In [ ]:
# Evaluate Q12 model on Q3 (expected: good) and Q4 (may degrade)
for name, X, y in [('Q3 (val)', X_q3, y_q3), ('Q4 (future)', X_q4, y_q4)]:
    proba = model_q12.predict_proba(X.values)[:, 1]
    auc = roc_auc_score(y.values, proba)
    rep = classification_report(y.values, (proba >= 0.5).astype(int), output_dict=True)
    print(f'\n--- {name} ---')
    print(f'  AUC-ROC:   {auc:.4f}')
    print(f'  Recall:    {rep["1"]["recall"]:.4f}')
    print(f'  Precision: {rep["1"]["precision"]:.4f}')
    print(f'  F1:        {rep["1"]["f1-score"]:.4f}')

In [ ]:
# Inject new fraud pattern into Q4 (ProductCD=W + high amount)
q4_drifted = inject_new_fraud_pattern(q4, amt_percentile=95,
                                       product_code='W', fraud_injection_rate=0.30)
X_q4d, y_q4d, _, _ = preprocess(q4_drifted, encoders=enc, medians=med, fit=False)

proba_d = model_q12.predict_proba(X_q4d.values)[:, 1]
auc_d = roc_auc_score(y_q4d.values, proba_d)
rep_d = classification_report(y_q4d.values, (proba_d >= 0.5).astype(int), output_dict=True)
print(f'\n--- Q4 WITH injected drift ---')
print(f'  AUC-ROC:   {auc_d:.4f}')
print(f'  Recall:    {rep_d["1"]["recall"]:.4f}')
print(f'  Precision: {rep_d["1"]["precision"]:.4f}')

In [ ]:
# Compute PSI and KS drift metrics between Q12 and Q4
os.makedirs('../outputs/drift', exist_ok=True)
df_drift = compute_drift(q12, q4, top_n=20, out_dir='../outputs/drift')
df_drift.head(10)

In [ ]:
# Retrain on Q4 data (simulates post-retrain recovery)
neg4, pos4 = (y_q4d == 0).sum(), (y_q4d == 1).sum()
model_q4 = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    scale_pos_weight=neg4/pos4, tree_method='hist',
    eval_metric='logloss', random_state=42, n_jobs=-1
)
model_q4.fit(X_q4d.values, y_q4d.values, verbose=50)

# Feature importance comparison
fi_q12 = dict(zip(X_train.columns, model_q12.feature_importances_))
fi_q4  = dict(zip(X_q4d.columns,  model_q4.feature_importances_))
fi_shift = compare_feature_importance(fi_q12, fi_q4, top_n=10, out_dir='../outputs/drift')
fi_shift.head(10)